# 07b - Building and Reading Hierarchical Clusters

K-means gave us a partition of the data for a requested number of clusters. Analysts might want more control, e.g., broad clusters for planning and finer distinctions for investigating workload. Today we will build a *hierarchy* of clusters that allows us to inspect cluster composition.

With this method the question shifts:

- from *how many clusters*
- to *which level of clustering would help the team investigate its work*.

A hierarchy supplies nested possibilities; analysts still need evidence before using them for decision support.

## References

- James et al., [*An Introduction to Statistical Learning with Applications in Python*](https://www.statlearning.com/), §12.4.2: hierarchical clustering, linkage, and reading dendrograms.
- [SciPy `linkage`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html): constructing the merge history and specifying linkage.
- [SciPy `dendrogram`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.dendrogram.html) and [cuts](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.cut_tree.html): displaying a hierarchy and extracting clusters.
- [Scikit-learn hierarchy guide](https://scikit-learn.org/stable/modules/clustering.html#hierarchical-clustering): linkage choices and Ward's squared-distance criterion.

## 1. Clusters Within Clusters

A **hierarchy** contains clusters within larger clusters. Here, the clusters are formed from distances between observations in a two-feature space.

The six observations are A=(0,0), B=(1,0), C=(3,1), D=(6,3), E=(10,0), and F=(11,1). The two features have comparable units, so we use these coordinates without scaling.

A **dendrogram** draws the hierarchy as a tree. Each leaf is an observation. Each join records two clusters becoming one. Look at the last join: which two clusters does it combine?

In [ ]:
from math import dist

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.cluster.hierarchy import cut_tree, dendrogram, fcluster, linkage
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

toy_names = np.array(list("ABCDEF"))
toy = np.array([[0, 0], [1, 0], [3, 1], [6, 3], [10, 0], [11, 1]], dtype=float)
preview_tree = linkage(toy, method="complete", metric="euclidean")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].scatter(toy[:, 0], toy[:, 1], s=70, color="#4477AA")
for name, point in zip(toy_names, toy, strict=True):
    axes[0].annotate(str(name), point, xytext=(5, 8),
                     textcoords="offset points")
axes[0].set(xlim=(-1, 12), ylim=(-1, 4), xlabel="Feature 1", ylabel="Feature 2",
            title="Six observations in two dimensions", aspect="equal")
axes[0].grid(alpha=0.2)
dendrogram(preview_tree, labels=toy_names.tolist(), ax=axes[1],
           color_threshold=0, above_threshold_color="#334155")
axes[1].set(ylabel="Complete-linkage distance", title="A hierarchy of these observations")
plt.show()

The last join combines {A,B,C} and {D,E,F}. Within those clusters, A and B join before C, while E and F join before D. The tree preserves those relationships at several levels. We will reconstruct the joins to understand their heights.

## 2. Build Up Through Merges

Hierarchical clustering generally uses either a top-down or bottom-up approach, using recursive splitting or joining, respectively. Formally, these are called divisive or agglomerative methods.

We will focus on the agglomerative clustering approach. It starts with one cluster per observation and builds larger clusters through merges. At each step, it joins the closest pair of current clusters until all observations are combined in a single cluster.

```text
Choose the feature representation, point distance, and linkage rule.
Start with one cluster per observation.
Repeat until only one cluster remains:
    Find the closest pair of current clusters.
    Join them and record the results, including the new cluster size.
    Update distances from the new cluster to the remaining clusters.
Return the merge history.
```

The feature coordinates and original point distances stay fixed. Cluster membership changes, so distances between current clusters must be updated. Each merge reduces the cluster count by one. Once joined, a cluster stays together in later merges.

This algorithm is *greedy*: it joins the closest available clusters under the chosen cluster-distance rule and never revisits that decision. Early merges cannot be undone, so later clusters depend on those local choices. A cut is not guaranteed to give the globally best partition for its cluster count.

## 3. SciPy

[SciPy](https://scipy.org/) is a scientific-computing library built on NumPy. It provides algorithms for tasks such as optimization, statistics, and clustering. This notebook uses its `scipy.cluster.hierarchy` module to build, draw, and cut a hierarchy.

The main packages have different jobs:

| Package or module | Role in this notebook |
| --- | --- |
| Python's `math` module | `dist()` calculates Euclidean distance between two points. |
| NumPy (`numpy`) | Stores feature coordinates and calculates arrays of cross-pair distances with `np.linalg.norm()`. |
| SciPy (`scipy.cluster.hierarchy`) | `linkage()` builds the merge history, `dendrogram()` draws it, and `cut_tree()` or `fcluster()` extracts cluster labels. |
| pandas (`pandas`) | Organizes observations, merge tables, and cluster profiles. |
| Matplotlib (`matplotlib.pyplot`) | Creates the scatterplots and comparison figures. |
| scikit-learn (`sklearn.preprocessing`) | `StandardScaler` learns feature scales and transforms the customer measurements. |

[Scikit-learn](https://scikit-learn.org/stable/getting_started.html) organizes much of its machine-learning workflow around fitted objects, such as `KMeans` and `StandardScaler`. The SciPy functions used here take arrays and return results directly. In particular, `linkage()` returns a merge-history array rather than a fitted estimator. We can pass that array to separate plotting and cut functions.

## 4. A Simple Hierarchy

Initially, every cluster contains one point. A and B are the closest pair, one unit apart, so they join first. How far is their new cluster {A,B} from C? There are two cross-pair distances to consider.

For points $P=(p_1,p_2)$ and $Q=(q_1,q_2)$, Euclidean distance combines the differences along both features:

$$
d(P,Q)=\sqrt{(p_1-q_1)^2+(p_2-q_2)^2}
$$

For A=(0,0) and C=(3,1), the distance is $\sqrt{(3-0)^2+(1-0)^2}=\sqrt{10}$, about 3.16. For B=(1,0) and C, it is $\sqrt{(3-1)^2+(1-0)^2}=\sqrt{5}$, about 2.24. Python's `math.dist()` takes two coordinate sequences and returns their Euclidean distance. We import it with `from math import dist`. The following cell calculates A-to-C and B-to-C separately, then displays them in a table.

In [ ]:
cross_distances = [
    dist(toy[0], toy[2]),  # A to C
    dist(toy[1], toy[2]),  # B to C
]
display(pd.DataFrame({"Cross-pair": ["A to C", "B to C"],
                      "Distance": cross_distances}).round(2))

A **linkage rule** defines how to summarize distances between members of different clusters. **Complete linkage** uses the largest cross-pair distance. For {A,B} and {C}, that gives

$$
D_{\mathrm{complete}}(\{A,B\},\{C\})=\max(\sqrt{10},\sqrt{5})=\sqrt{10}\approx 3.16
$$

The value 3.16 comes from A to C. The point distances have not changed; we have chosen how to turn them into one cluster distance.

This cluster distance is the **merge height**: the value at which the two clusters join, shown vertically in the dendrogram. Here, {A,B} joins C at a height of about 3.16.

Other linkage rules answer different questions about cluster closeness:

- **Single linkage** uses the nearest cross-pair. It can reveal connected or elongated patterns, although a few short connections can create long chains.
- **Average linkage** uses the mean of all cross-pair distances. It suits a comparison based on overall closeness rather than one nearest or farthest pair.
- **Ward linkage** chooses the merge with the smallest increase in within-cluster squared distance. It is useful when compact clusters are the goal and Euclidean geometry is appropriate.

### 4.1 Pause: Which Pair Joins Next?

Does {A,B} join C next? Compare its complete-linkage distance of about 3.16 with the remaining singleton pairs. Then explain which quantities must be updated after that next merge.

##### Answer

E and F join next because their distance is $\sqrt{2}$, about 1.41, smaller than 3.16. Distances involving the new cluster {E,F} must be updated. Its cross-pair distances to D are $\sqrt{(10-6)^2+(0-3)^2}=5$ and $\sqrt{(11-6)^2+(1-3)^2}=\sqrt{29}$, about 5.39. Complete linkage uses the larger value, 5.39. Distances among unchanged clusters stay the same.

### 4.2 Continue the Merge History

E and F join at about 1.41 before {A,B} joins C at about 3.16. After those merges, the current clusters are {A,B,C}, {D}, and {E,F}. Compare the three remaining candidate joins.

In [ ]:
current_clusters = {"ABC": [0, 1, 2], "D": [3], "EF": [4, 5]}
candidate_pairs = [("ABC", "D"), ("ABC", "EF"), ("D", "EF")]
candidate_distances = []
for left, right in candidate_pairs:
    left_points = toy[current_clusters[left]]
    right_points = toy[current_clusters[right]]
    cross_pairs = left_points[:, None, :] - right_points[None, :, :]
    distance = np.linalg.norm(cross_pairs, axis=2).max()
    candidate_distances.append({"Clusters": f"{left} and {right}",
                                "Complete-linkage distance": distance})
display(pd.DataFrame(candidate_distances).round(2))

D joins {E,F} at about 5.39. The final join combines {A,B,C} and {D,E,F} at $\sqrt{122}$, about 11.05, the A-to-F distance. The largest cross-pair determines each complete-linkage merge height; it does not mean every pair in the resulting cluster has that distance.

### 4.3 From Merges to SciPy

Last time, scikit-learn's `fit()` stored results in a fitted K-means object. Here, SciPy's `linkage()` directly returns an array describing the merge history. We give it one row per observation and explicitly choose complete linkage with Euclidean point distance. SciPy's default linkage is single, so the selected rule belongs in the call.

In [ ]:
toy_tree = linkage(toy, method="complete", metric="euclidean")
merge_matrix = pd.DataFrame(toy_tree, columns=["Child 1", "Child 2", "Height", "Size"])
merge_matrix = merge_matrix.astype({"Child 1": int, "Child 2": int, "Size": int})
merge_matrix.index = pd.Index(range(6, 11), name="New cluster ID")
display(merge_matrix.round(2))

Each row records one merge: its two children, its height, and the number of original observations in the union. IDs 0–5 refer to A–F. New cluster IDs begin at six, so the first row creates cluster 6 from children 0 and 1, at height one, with size two. Later rows can refer to those new clusters.

The following table replaces the numeric child IDs with observation names. Match its third join to the {A,B}-to-C calculation.

In [ ]:
members = {i: str(name) for i, name in enumerate(toy_names)}
merge_rows = []
for step, (left, right, height, size) in enumerate(toy_tree):
    left_names, right_names = members[int(left)], members[int(right)]
    merge_rows.append({"Join": step + 1, "Children": f"{left_names} + {right_names}",
                       "Height": height, "Observations": int(size)})
    members[len(toy) + step] = "".join(sorted(left_names + right_names))
display(pd.DataFrame(merge_rows).round(2))

## 5. Read Heights and Choose a Cut

`dendrogram()` draws the merge-history array. The height of a join reports the linkage value used there. To compare two leaves in the tree, follow their branches to the first join containing both. A and C first meet at about 3.16. D and E first meet at about 5.39, although their original point distance is five.

A horizontal **cut** chooses a level of clustering. At height four, the tree has three clusters: {A,B,C}, {D}, and {E,F}. At height seven, it has two: {A,B,C} and {D,E,F}. Count the branches crossing each line.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
dendrogram(toy_tree, labels=toy_names.tolist(), ax=ax, color_threshold=0,
           above_threshold_color="#334155")
ax.axhline(4, color="#b45309", linestyle="--", label="3 clusters: height 4")
ax.axhline(7, color="#2563eb", linestyle="--", label="2 clusters: height 7")
ax.set(title="Two cuts of the same hierarchy", ylim=(0, 12), xlabel="Observation",
       ylabel="Complete-linkage distance")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=8)
plt.show()

Horizontal proximity is drawing layout. Read relationships by following branches, rather than measuring the spacing between leaves.

### 5.1 Extract Membership Labels

`fcluster()` extracts labels using a distance threshold. `cut_tree()` can instead request a number of clusters. Its result has one column per requested cut; `.ravel()` converts a single column to a label vector. Both operate on the existing merge history without fitting another tree.

The merge-history table assigns an ID to every cluster created during merging. The table below instead gives each observation a **membership label for a particular cut**. These labels are assigned afresh and do not refer to merge-history IDs. Matching labels within a column mean that observations belong to the same cluster.

In [ ]:
labels_at_four = fcluster(toy_tree, t=4, criterion="distance")
labels_at_seven = fcluster(toy_tree, t=7, criterion="distance")
toy_three = cut_tree(toy_tree, n_clusters=3).ravel()
display(pd.DataFrame({"Observation": toy_names, "Label at Height 4": labels_at_four,
                      "Label at Height 7": labels_at_seven, "Label for 3-Cluster Cut": toy_three}))

At height 4, A, B, and C all have label 1, so they form one cluster. This is the cluster with ID 8 in the merge history. Label 1 does not refer to the earlier cluster {A, B} or to a position in the dendrogram.

The height-4 cut and the three-cluster cut produce the same member sets, even though their label numbers differ. Cluster labels are arbitrary identifiers; compare member sets rather than label numbers across cuts.

With tied merge heights, some counts cannot be selected by a single horizontal threshold; a count-based request follows the merge sequence instead.

### 5.2 Pause: What Does a Cut Establish?

Explain how the three-cluster cut fits inside the two-cluster cut. Would the long interval before the last join establish that these observations come from two real populations?

##### Answer

{A,B,C} stays together, while {D} and {E,F} combine into {D,E,F}. These partitions are nested: each finer cluster lies inside one coarser cluster from the same tree.

The two-cluster partition persists for every cut between about 5.39 and 11.05. This wide gap suggests a useful level to inspect, but choosing it still requires a reason to prefer two clusters for the task.

### 5.3 Other Ways to Extract Clusters

The `criterion` argument controls how `fcluster()` extracts clusters from the same tree. It also changes what `t` means. The [SciPy reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.fcluster.html) lists five options:

| Criterion | How it chooses clusters |
| --- | --- |
| `distance` | Cut at the merge-height threshold `t`, as above. |
| `maxclust` | Find the smallest height threshold that produces at most `t` clusters. |
| `inconsistent` (default) | Apply threshold `t` to a measure of how unusual a merge height is relative to nearby merges below it. |
| `monocrit` | Apply threshold `t` to a supplied statistic that never decreases as we move up the tree. |
| `maxclust_monocrit` | Choose a threshold on that supplied statistic to produce at most `t` clusters. |

Use `distance` when a height threshold matters and `maxclust` when a cluster-count limit matters. The other options require a local inconsistency measure or a custom statistic; we will leave their details for further study.

This brings us back to K-means: we can ask for a cluster count instead of choosing a distance threshold. K-means fits centroids for a chosen $k$. Here, `maxclust` extracts at most $k$ clusters from an already-built hierarchy; changing the requested count changes the cut, without refitting the tree. The resulting partitions remain nested, whereas separate K-means fits at different values of $k$ need not be nested.

Unlike K-means, agglomerative clustering has no random starting centers. With fixed data and row order, representation, distance, linkage, and implementation, this workflow reproduces the same hierarchy. Tied distances can admit different merge orders, so tie handling also matters.

In [ ]:
labels_max_three = fcluster(toy_tree, t=3, criterion="maxclust")
display(pd.DataFrame({"Observation": toy_names,
                      "Label at Height 4": labels_at_four,
                      "Label for At Most 3 Clusters": labels_max_three}))

Here, `t=3` requests at most three clusters, rather than a height of three. The result is {A,B,C}, {D}, and {E,F}, matching the height-4 cut. No new tree is fitted.

**At most** matters: tied merge heights can make the count jump past the requested number, so `maxclust` may return fewer clusters. In contrast, `cut_tree(..., n_clusters=3)` selects three clusters by following the merge sequence. As before, compare memberships rather than label numbers.

Compare the three-cluster memberships on the same six observations. Both panels use the original coordinates and matching colors for matching member sets.

In [ ]:
toy_kmeans = KMeans(n_clusters=3, n_init=10, random_state=7130)
toy_kmeans_labels = toy_kmeans.fit_predict(toy)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained",
                         sharex=True, sharey=True)
colors = ["#4477AA", "#EE7733", "#228833"]
for ax, labels, title in zip(
    axes, [toy_kmeans_labels, labels_max_three],
    ["K-means: k = 3", "Complete linkage: cut to 3 clusters"], strict=True
):
    cluster_ids = sorted(np.unique(labels),
                         key=lambda label: np.flatnonzero(labels == label)[0])
    for color, cluster_id in zip(colors, cluster_ids, strict=True):
        mask = labels == cluster_id
        member_names = ", ".join(toy_names[mask])
        ax.scatter(toy[mask, 0], toy[mask, 1], s=75, color=color,
                   label="{" + member_names + "}")
    for name, point in zip(toy_names, toy, strict=True):
        ax.annotate(str(name), point, xytext=(5, 8), textcoords="offset points")
    ax.set(title=title, xlim=(-1, 12), ylim=(-1, 4),
           xlabel="Feature 1", aspect="equal")
    ax.grid(alpha=0.2)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=3, fontsize=8)
axes[0].set_ylabel("Feature 2")
plt.show()

Both methods give {A,B,C}, {D}, and {E,F} for these toy points. Agreement here does not mean they will agree on other data: K-means fits centroids to reduce within-cluster squared distance, while complete linkage builds a hierarchy through its merge rule.

**Hierarchical clustering has no built-in decision boundary.** Cutting this tree assigns the observed points to clusters but does not specify how to assign a new point. K-means can assign a new point to its nearest fitted centroid; doing so for hierarchical clusters would require an additional assignment rule. These panels show observed memberships, rather than decision boundaries.

## 6. Other Definitions of Close Clusters

For {A,B} and {C}, the cross-pair distances remain $\sqrt{10}$ and $\sqrt{5}$. Single linkage uses their minimum, about 2.24. Average linkage uses their mean, about 2.70. Complete linkage uses their maximum, about 3.16. Changing linkage changes the meaning of a candidate join, even with the same observations and point distances.

| Linkage | Criterion for joining two clusters |
| --- | --- |
| Single | Smallest distance between a member of each cluster. |
| Complete | Largest distance between a member of each cluster. |
| Average | Mean of all cross-pair distances. |
| Ward | Smallest increase in total within-cluster squared distance after a merge. |

**Ward linkage** connects to the squared-distance objective from K-means. It chooses the available merge that adds the least within-cluster squared distance, using Euclidean geometry. K-means can reassign points while updating centers for a fixed K; Ward joins existing clusters irreversibly. Their partitions need not agree.

For single linkage, one short connection can join two clusters even when their other members are far apart. This can create chains. Compare the same six observations at two clusters under single and complete linkage.

In [ ]:
toy_comparisons = []
for method in ("single", "complete"):
    tree = linkage(toy, method=method, metric="euclidean")
    labels = cut_tree(tree, n_clusters=2).ravel()
    clusters = ["".join(toy_names[labels == cluster]) for cluster in np.unique(labels)]
    toy_comparisons.append({"Linkage": method, "Two clusters": " / ".join(clusters)})
display(pd.DataFrame(toy_comparisons))

Single yields ABCD/EF, while complete yields ABC/DEF. This is a different hierarchy caused by a different rule. Neither partition becomes useful solely because the algorithm produced it. Merge heights also depend on the rule, so one numerical threshold is not a common comparison across all linkage choices.

## 7. Return to the Customers

We will reuse the same 72 simulated customers, with request count and average minutes per request. The two attributes were generated independently, with no planted customer types. Standardizing preserves the feature space used in the preceding customer analysis.

The scaler remains a fitted scikit-learn object. Its `transform()` supplies the standardized rows to SciPy; `linkage()` then returns the merge history. Requesting a full hierarchy postpones the choice of a flat partition until we inspect a cut.

In [ ]:
rng = np.random.default_rng(7130)
customers = pd.DataFrame({"requests": rng.integers(1, 13, size=72),
                          "avg_minutes": rng.integers(8, 65, size=72)},
                         index=pd.Index(range(1, 73), name="customer"))
scaler = StandardScaler()
scaler.fit(customers)
scaled_customers = scaler.transform(customers)
customer_tree = linkage(scaled_customers, method="complete", metric="euclidean")
display(customers.head())
print(f"Merge-history shape: {customer_tree.shape}")

The 71 rows record the merges needed to combine 72 observations. A full plot would contain 72 leaves. The overview below collapses smaller branches into 12 terminals so the upper joins are readable. Parenthesized numbers count customers inside those collapsed branches; they are not customer IDs.

The two cut heights are chosen between the relevant merges to illustrate two and three clusters. They are not optimized thresholds. All cuts use the full merge-history array.

In [ ]:
customer_cut_two = customer_tree[-2:, 2].mean()
customer_cut_three = customer_tree[-3:-1, 2].mean()
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
dendrogram(customer_tree, truncate_mode="lastp", p=12, show_leaf_counts=True,
           ax=ax, color_threshold=0, above_threshold_color="#334155")
for height, k, color in ((customer_cut_two, 2, "#2563eb"),
                         (customer_cut_three, 3, "#b45309")):
    ax.axhline(height, color=color, linestyle="--", label=f"{k} clusters: height {height:.2f}")
ax.set(title="Complete-linkage customer hierarchy: collapsed branch overview",
       ylabel="Complete-linkage distance",
       xlabel="Parentheses count customers in collapsed branches")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=9)
plt.show()

### 7.1 What Does a Finer Cut Add?

Cut the same customer tree at two and three clusters. The following figures display memberships in original units; the distances that built the tree used standardized values. Read the member sets and profiles before attaching service meaning to them.

In [ ]:
customer_two = cut_tree(customer_tree, n_clusters=2).ravel()
customer_three = cut_tree(customer_tree, n_clusters=3).ravel()

def profiles_for(labels):
    return (customers.assign(cluster=labels).groupby("cluster")
            .agg(customers=("requests", "size"),
                 mean_requests=("requests", "mean"),
                 mean_minutes=("avg_minutes", "mean")))

display(pd.concat({"Two clusters": profiles_for(customer_two),
                   "Three clusters": profiles_for(customer_three)}, names=["Cut"]).round(2))

The two panels below show which members remain together and which cluster separates at the finer cut. Their axes are shared. Read the cluster counts alongside the profile table.

In [ ]:
def plot_customer_clusters(ax, labels, title):
    colors = ("#4477AA", "#EE6677", "#228833")
    markers = ("o", "^", "s")
    for cluster in np.unique(labels):
        rows = customers.iloc[np.flatnonzero(labels == cluster)]
        ax.scatter(rows["requests"], rows["avg_minutes"], s=38, alpha=0.8,
                   color=colors[int(cluster)], marker=markers[int(cluster)],
                   label=f"Cluster {cluster}: n={len(rows)}")
    ax.set(title=title, xlabel="Requests last term", xlim=(0, 13), ylim=(0, 70))
    ax.grid(alpha=0.15)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharex=True, sharey=True,
                         layout="constrained")
plot_customer_clusters(axes[0], customer_two, "Two clusters from the tree")
plot_customer_clusters(axes[1], customer_three, "Three clusters from the same tree")
axes[0].set_ylabel("Average minutes per request")
fig.suptitle("Same complete-linkage hierarchy; different cut")
plt.show()

The three-cluster cut retains 36 customers with shorter requests, averaging 22.25 minutes. It separates the other customers into 25 averaging nine requests and 44.4 minutes, and 11 averaging 2.27 requests and 50.36 minutes. The finer cut distinguishes request frequency among customers with longer requests.

#### 7.1.1 Pause: Is the Additional Cluster Useful?

What could the extra distinction change for the support team's investigation? Would you use two clusters or three for an initial discussion, and what would you check before recommending a change?

##### Answer

The two-cluster cut could frame an investigation of shorter versus longer requests. The third cluster asks whether frequent customers with longer requests need different support from infrequent customers with longer requests. Either resolution can supply a question; neither supplies a service policy.

We would inspect request content and ask the team which differences affect its work. Another period of real customer records would test whether the distinction recurs. These independently generated measurements cannot establish real customer types or their needs.

### 7.2 Change the Linkage Rule

Keep the customers, standardized coordinates, Euclidean distance, and three-cluster request fixed. Change only complete linkage to single linkage. Predict whether joining through one short connection will produce similar cluster sizes.

In [ ]:
single_customer_tree = linkage(scaled_customers, method="single", metric="euclidean")
single_customer_three = cut_tree(single_customer_tree, n_clusters=3).ravel()
display(pd.concat({"Complete": profiles_for(customer_three),
                   "Single": profiles_for(single_customer_three)}, names=["Linkage"]).round(2))

The following panels show the two partitions on common original-unit axes. Each fit uses standardized distances; only the linkage rule changes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharex=True, sharey=True,
                         layout="constrained")
plot_customer_clusters(axes[0], customer_three, "Complete linkage")
plot_customer_clusters(axes[1], single_customer_three, "Single linkage")
axes[0].set_ylabel("Average minutes per request")
fig.suptitle("Same standardized customers and three clusters; change only linkage")
plt.show()

Complete produces sizes 36, 25, and 11. Single produces one cluster of 68 and two clusters of two. A sequence of short connections can join most of the observations while leaving a few small clusters. Cluster numbers and colors are local to each panel.

Single linkage can be useful when connected structure is the question. For this service investigation, the large connected cluster gives few distinctions among most customers. Choose the rule with its meaning and purpose in view, rather than assuming one rule is universally best.

## 8. Use a Hierarchy With a Purpose

This workflow builds a hierarchy, extracts a cut, and inspects its existing members. It does not provide K-means' nearest-center prediction for a later customer. Scikit-learn's `AgglomerativeClustering` also has no built-in `predict()` or `transform()`. Assigning future customers would require a separately chosen rule or refitting and reassessing the hierarchy.

A hierarchy gives nested clusterings under chosen features, scale, point distance, and linkage. Reading the merge history explains how they formed; choosing a cut determines which resolution we inspect. Their usefulness depends on the decision and on evidence beyond the fitted tree.

### 8.1 Pause: What Would You Take to the Team?

Propose one customer clustering for further investigation. Explain the linkage and resolution you chose, identify evidence that could change your choice, and state one conclusion these simulated records cannot support.

##### Answer

The three-cluster complete-linkage cut could support a discussion of request frequency among customers with longer requests. Its profiles make that distinction visible. We should justify feature weighting and inspect another period of real records.

The team may find that request content matters more than these two measurements, or that a broad two-cluster division is sufficient. These results do not establish stable customer types or justify changing support. They identify questions and the evidence needed to answer them.